In [ ]:
pip install scikit-surprise

In [ ]:
pip install surprise

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2357254 sha256=8fdcd4bc1b0ce6d672ef5354ec676fd82b27faaa0b2ecbc16adc92a02ea88bb5
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise


In [ ]:
import pandas as pd
import surprise

In [ ]:
ordermasters = pd.read_csv('/content/orderMasters.csv')
orderdetails = pd.read_csv('/content/orderDetails.csv')
products = pd.read_csv('/content/products (2).csv')
ingredients = pd.read_csv('/content/Bản CSV của HMTPTKD-Database - ingredients.csv')
productcategories = pd.read_csv('/content/productCategories.csv')
ingredientcategories = pd.read_csv('/content/Bản sao của HMTPTKD-Database - ingredientCategories.csv')
recipes = pd.read_csv('/content/Bản sao của HMTPTKD-Database - recipes.csv')

In [ ]:
df_merged = orderdetails.merge(ordermasters, how="left", on='OrderID')
df_merged.drop(df_merged[df_merged["Order_Status"] == "Cancelled"].index, inplace=True)

In [ ]:
df_merged

,OrderDetailID,OrderID,ProductID,Product_Qty,Review_Star,PriceChange,CustomerID,Order_Date,Order_Status,EmployeeID,ManagerID,Order_TimeStart,Order_TimeEnd
0,1,1,P10M,9,1,0,C11029,6/2/2022,Finished,108,1,6:00:00,6:10:00
1,2,1,P11L,10,3,0,C11029,6/2/2022,Finished,108,1,6:00:00,6:10:00
2,3,1,P11M,8,4,0,C11029,6/2/2022,Finished,108,1,6:00:00,6:10:00
3,4,1,P7L,1,4,0,C11029,6/2/2022,Finished,108,1,6:00:00,6:10:00
4,5,1,P7M,8,2,0,C11029,6/2/2022,Finished,108,1,6:00:00,6:10:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
301970,301971,80496,P4M,9,1,0,C12658,7/30/2023,Finished,111,1,21:44:00,21:55:00
301971,301972,80496,P17M,5,3,0,C12658,7/30/2023,Finished,111,1,21:44:00,21:55:00
301972,301973,80497,P2L,6,5,0,C13471,7/30/2023,Finished,123,1,21:53:00,22:02:00
301973,301974,80497,P10L,8,1,0,C13471,7/30/2023,Finished,123,1,21:53:00,22:02:00


In [ ]:
df_merged.drop(["OrderDetailID", "OrderID", "Order_Date", "Order_Status", "EmployeeID", "ManagerID", "Order_TimeStart", "Order_TimeEnd", "OrderDetailID","Product_Qty", "PriceChange"], axis=1, inplace=True)

In [ ]:
df = df_merged.merge(products[['ProductID', "ProductCategoryID", "Product_Name", "Product_Size", "Product_UnitPrice"]], on="ProductID")

In [ ]:
df

,ProductID,Review_Star,CustomerID,ProductCategoryID,Product_Name,Product_Size,Product_UnitPrice
0,P10M,1,C11029,PC1,Hot Chocolate,M,18
1,P10M,5,C11002,PC1,Hot Chocolate,M,18
2,P10M,3,C11009,PC1,Hot Chocolate,M,18
3,P10M,3,C11095,PC1,Hot Chocolate,M,18
4,P10M,2,C11011,PC1,Hot Chocolate,M,18
...,...,...,...,...,...,...,...
301928,P18M,5,C12468,PC3,Vanilla Bean Frappuccino,M,25
301929,P18M,5,C12433,PC3,Vanilla Bean Frappuccino,M,25
301930,P18M,4,C12906,PC3,Vanilla Bean Frappuccino,M,25
301931,P18M,3,C12428,PC3,Vanilla Bean Frappuccino,M,25


In [ ]:
# dic = {"PC1": "Coffee", "PC2": "Latte", "PC3": "Frappuccino"}

In [ ]:
# df["CategoryName"] = df['ProductCategoryID'].map(dic)

In [ ]:
# df.drop(["ProductID", "ProductCategoryID"], axis=1, inplace=True)

In [ ]:
df

,ProductID,Review_Star,CustomerID,ProductCategoryID,Product_Name,Product_Size,Product_UnitPrice
0,P10M,1,C11029,PC1,Hot Chocolate,M,18
1,P10M,5,C11002,PC1,Hot Chocolate,M,18
2,P10M,3,C11009,PC1,Hot Chocolate,M,18
3,P10M,3,C11095,PC1,Hot Chocolate,M,18
4,P10M,2,C11011,PC1,Hot Chocolate,M,18
...,...,...,...,...,...,...,...
301928,P18M,5,C12468,PC3,Vanilla Bean Frappuccino,M,25
301929,P18M,5,C12433,PC3,Vanilla Bean Frappuccino,M,25
301930,P18M,4,C12906,PC3,Vanilla Bean Frappuccino,M,25
301931,P18M,3,C12428,PC3,Vanilla Bean Frappuccino,M,25


In [ ]:
from surprise import Dataset, Reader, SVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [ ]:
content_df = df[['ProductID', 'Product_Name', 'Product_Size', "Product_UnitPrice", "ProductCategoryID"]]

In [ ]:
df['Content'] =  content_df.apply(lambda row: ' '.join(row.dropna().astype(str)), axis=1)

In [ ]:
df['Content']

0                    P10M Hot Chocolate M 18 PC1
1                    P10M Hot Chocolate M 18 PC1
2                    P10M Hot Chocolate M 18 PC1
3                    P10M Hot Chocolate M 18 PC1
4                    P10M Hot Chocolate M 18 PC1
                           ...                  
301928    P18M Vanilla Bean Frappuccino M 25 PC3
301929    P18M Vanilla Bean Frappuccino M 25 PC3
301930    P18M Vanilla Bean Frappuccino M 25 PC3
301931    P18M Vanilla Bean Frappuccino M 25 PC3
301932    P18M Vanilla Bean Frappuccino M 25 PC3
Name: Content, Length: 301933, dtype: object

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
content_matrix = tfidf_vectorizer.fit_transform(df['Content'])

content_similarity = linear_kernel(content_matrix, content_matrix)

In [ ]:
  df_colab_filter = df[['CustomerID', "Product_Name", "Review_Star"]]

In [ ]:
df_colab_filter

,CustomerID,Product_Name,Review_Star
0,C11029,Hot Chocolate,1
1,C11002,Hot Chocolate,5
2,C11009,Hot Chocolate,3
3,C11095,Hot Chocolate,3
4,C11011,Hot Chocolate,2
...,...,...,...
301928,C12468,Vanilla Bean Frappuccino,5
301929,C12433,Vanilla Bean Frappuccino,5
301930,C12906,Vanilla Bean Frappuccino,4
301931,C12428,Vanilla Bean Frappuccino,3


In [ ]:
def get_content_based_recommendations(product_name, top_n):
    # Check if the product_name exists in the DataFrame
    matching_rows = content_df[content_df['ProductName'] == product_name]

    if matching_rows.empty:
        raise ValueError(f"ProductName {product_name} not found in content_df.")

    index = matching_rows.index[0]
    similarity_scores = content_similarity[index]
    similar_indices = similarity_scores.argsort()[::-1][1:top_n + 1]

    recommended_products = content_df.iloc[similar_indices]['ProductName']
    return recommended_products.tolist()

# Example call
try:
    recommendations = get_content_based_recommendations('P13M', 10)
    print(recommendations)
except ValueError as e:
    print(e)
